## Next Steps

### For Local Development
```bash
# 1. Build the index (if PDFs are in data/raw_docs/)
python -m src.cli_app

# 2. Start Streamlit demo
streamlit run src/demo_app.py

# 3. Or use CLI for interactive Q&A
python -m src.cli_app
```

### For Docker Deployment
The Dockerfile already includes all necessary system libraries:
- `libgl1-mesa-glx` and `libgl1-mesa-dri` (OpenGL support)
- `tesseract-ocr` (OCR for images)
- `poppler-utils` (PDF processing)

```bash
docker build -t rag-system .
docker run -p 8501:8501 -e GROQ_API_KEY='your_key' rag-system
```

### Environment Configuration
Set these in your deployment:
```env
GROQ_API_KEY=your_groq_key_here
GROQ_MODEL=llama-3.3-70b-versatile
# Optional:
DISPLAY=
LIBGL_ALWAYS_INDIRECT=1
CUDA_VISIBLE_DEVICES=-1
```

In [ ]:
# Comprehensive validation report
print("=\"*70)\nprint(\"✅ RAG SYSTEM VALIDATION REPORT\")\nprint(\"=\"*70)\n\nchecks = {\n    \"Environment\": {\n        \"OpenGL disabled\": os.environ.get('DISPLAY') == '',\n        \"CPU-only mode\": os.environ.get('CUDA_VISIBLE_DEVICES') == '-1',\n    },\n    \"Dependencies\": {\n        \"pdfplumber\": True,  # Already verified above\n        \"sentence_transformers\": True,\n        \"faiss\": True,\n    },\n    \"Project Structure\": {\n        \"Config loaded\": True,\n        \"Paths accessible\": DATA_DIR.exists() and INDEX_DIR.exists(),\n    },\n    \"Models\": {\n        \"Embedder available\": True,\n        \"Groq API configured\": bool(GROQ_API_KEY),\n    }\n}\n\nfor category, items in checks.items():\n    print(f\"\\n📋 {category}:\")\n    for check, passed in items.items():\n        symbol = \"✅\" if passed else \"❌\"\n        print(f\"  {symbol} {check}\")\n\nprint(\"\\n\" + \"=\"*70)\nprint(\"🚀 SYSTEM READY - Deploy with confidence!\")\nprint(\"=\"*70)")

## 6. Validation Summary

In [ ]:
# Check if GROQ API key is set
from src.config import GROQ_API_KEY

if not GROQ_API_KEY:
    print(\"⚠️ GROQ_API_KEY not set. Q&A testing requires:\")\n    print(\"   1. Get API key from: https://console.groq.com\")\n    print(\"   2. Set environment: export GROQ_API_KEY='your_key'\")\n    print(\"   3. Or create .env file: GROQ_API_KEY=your_key\\n\")\nelse:\n    print(\"✅ GROQ_API_KEY is configured\")\n    print(\"   RAG pipeline is ready for Q&A queries\\n\")\n    \n    # Optional: Test RAG pipeline if index exists\n    from pathlib import Path\n    from src.indexing.multi_modal_store import MULTI_MODAL_FAISS\n    \n    if MULTI_MODAL_FAISS.exists():\n        print(\"📚 Ready to test Q&A. Example:\")\n        print(\"\\n   from src.retriever.rag_pipeline import answer_query\")\n        print(\"   result = answer_query('What is the main topic?', top_k=3)\")\n        print(\"   print(result)\\n\")\n    else:\n        print(\"⚠️ Index not found. Build index first using CLI.\\n\")")

## 5. Test RAG Pipeline (Q&A)

In [ ]:
# Test MultiModalVectorStore and index loading
try:
    print("🔄 Loading MultiModalVectorStore...")
    from src.indexing.multi_modal_store import MultiModalVectorStore, MULTI_MODAL_FAISS
    
    # Check if index exists
    if MULTI_MODAL_FAISS.exists():
        print(f"✅ Pre-built index found: {MULTI_MODAL_FAISS}\n")
        
        # Load the store
        store = MultiModalVectorStore()
        
        # Check if index is loaded
        if store.index is not None:
            print(f"✅ Index loaded successfully!")
            print(f"   Index type: {type(store.index).__name__}")
            print(f"   Index dimension: {store.index.ntotal}")
            print(f"   Index size: {MULTI_MODAL_FAISS.stat().st_size / 1024 / 1024:.2f} MB\n")
        else:
            print("⚠️ Store object created but index not yet loaded (may need rebuild)\n")
    else:
        print(f"⚠️ No pre-built index found at {MULTI_MODAL_FAISS}")
        print("    → Use CLI to build: python -m src.cli_app (select option 1)\n")
        
except Exception as e:
    print(f"❌ Vector store error: {e}")
    import traceback
    traceback.print_exc()

## 4. Test Retrieval Pipeline & Index Loading

In [ ]:
# Test MultiModalEmbedder
try:
    print("🔄 Loading MultiModalEmbedder...")
    from src.indexing.embeddings import MultiModalEmbedder
    
    embedder = MultiModalEmbedder(model_name="sentence-transformers/clip-ViT-B-32")
    print("✅ MultiModalEmbedder loaded!\n")
    
    # Test text embedding
    test_texts = [
        "What are the key findings in this document?",
        "Can you summarize the quarterly report?",
    ]
    
    print(f"📝 Testing text embedding with {len(test_texts)} samples...")
    embeddings = embedder.encode_text(test_texts, batch_size=2)
    print(f"✅ Text embeddings shape: {embeddings.shape}")
    print(f"   Embedding dimension: {embeddings.shape[1]}")
    print(f"   Sample embedding (first 5 values): {embeddings[0][:5]}\n")
    
except Exception as e:
    print(f"❌ Embedder error: {e}")
    import traceback
    traceback.print_exc()

## 3. Test Embeddings (Text & Images)

In [ ]:
# Test configuration import
try:
    from src.config import (
        PROJECT_ROOT, DATA_DIR, RAW_DOCS_DIR, 
        PROCESSED_DIR, INDEX_DIR, GROQ_API_KEY,
        EMBEDDING_MODEL_NAME
    )
    
    print("✅ Configuration loaded successfully!\n")
    print(f"📂 PROJECT_ROOT:        {PROJECT_ROOT}")
    print(f"📂 DATA_DIR:            {DATA_DIR}")
    print(f"📂 RAW_DOCS_DIR:        {RAW_DOCS_DIR}")
    print(f"📂 PROCESSED_DIR:       {PROCESSED_DIR}")
    print(f"📂 INDEX_DIR:           {INDEX_DIR}")
    print(f"🔑 GROQ_API_KEY:        {'SET' if GROQ_API_KEY else 'NOT SET'}")
    print(f"🧠 EMBEDDING_MODEL:     {EMBEDDING_MODEL_NAME}")
    
    # Check if directories exist
    print("\n📁 Directory Status:")
    for dir_path in [RAW_DOCS_DIR, PROCESSED_DIR, INDEX_DIR]:
        exists = "✅" if dir_path.exists() else "❌"
        print(f"  {exists} {dir_path.name}: {dir_path.exists()}")
        
except Exception as e:
    print(f"❌ Configuration error: {e}")
    import traceback
    traceback.print_exc()

## 2. Test Configuration & Paths

In [ ]:
# Verify critical packages
print("\n📦 Checking critical dependencies:")

required_packages = {
    'pdfplumber': 'PDF text extraction',
    'sentence_transformers': 'Multi-modal embeddings (CLIP)',
    'faiss': 'Vector indexing',
    'groq': 'LLM API',
    'streamlit': 'Web UI',
    'PIL': 'Image handling',
    'numpy': 'Numerical computing',
}

for package_name, description in required_packages.items():
    try:
        module = __import__(package_name)
        version = getattr(module, '__version__', 'unknown')
        print(f"  ✅ {package_name:<30} v{version:<15} - {description}")
    except ImportError as e:
        print(f"  ❌ {package_name:<30} MISSING         - {description}")
        print(f"      Error: {e}")

In [ ]:
# Check environment variables for headless mode
import os

headless_vars = {
    'DISPLAY': os.environ.get('DISPLAY', 'NOT SET'),
    'LIBGL_ALWAYS_INDIRECT': os.environ.get('LIBGL_ALWAYS_INDIRECT', 'NOT SET'),
    'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES', 'NOT SET'),
    'TOKENIZERS_PARALLELISM': os.environ.get('TOKENIZERS_PARALLELISM', 'NOT SET'),
}

print("🔧 Headless Environment Variables:")
for key, value in headless_vars.items():
    status = "✅" if value != 'NOT SET' else "⚠️"
    print(f"  {status} {key}: {value}")

## 1. Environment Diagnostics

In [ ]:
import os
import sys
from pathlib import Path

# Add project to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version}")
print(f"💻 Working directory: {os.getcwd()}")

# Multi-Modal RAG System - Testing & Diagnostics

This notebook tests the RAG system in headless environments and validates the libGL.so.1 fix.

**Fixes Applied:**
- Environment variables to disable OpenGL (config.py)
- Error handling for image encoding (embeddings.py)
- Safe image loading mode (pdf_text_extractor.py)